# 01 — Data Exploration

Explore the WiDS Datathon 2020 ICU mortality dataset (or a synthetic
schema-compatible fallback if Kaggle credentials are not configured).

**Goal:** understand class balance, missingness, and which features are
most strongly associated with ``hospital_death`` — these will drive
feature selection for the quantum encoding.

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from qml_healthcare.config import CATEGORICAL_FEATURES, FIGURES_DIR, NUMERIC_FEATURES, TARGET
from qml_healthcare.data.download import ensure_dataset
from qml_healthcare.data.loader import load_raw
from qml_healthcare.evaluation import figure_path, plot_class_balance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

In [ ]:
csv_path = ensure_dataset()
df = load_raw(csv_path)
print(f"Dataset: {csv_path}")
print(f"Shape: {df.shape}")
df.head()

## Class balance

ICU mortality is highly imbalanced — typically ~8% positive in WiDS.
Any honest evaluation needs balanced accuracy, ROC-AUC, and PR-AUC,
not just raw accuracy.

In [ ]:
print("Class counts:", df[TARGET].value_counts().to_dict())
print(f"Positive rate: {df[TARGET].mean():.3%}")
plot_class_balance(df[TARGET].to_numpy(), figure_path("class_balance.png"))

## Missingness pattern

Median imputation is reasonable for the curated subset; we visualize
missingness so the choice is justified and reproducible.

In [ ]:
feature_cols = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c in df.columns]
missing = df[feature_cols].isna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(missing))))
sns.barplot(x=missing.values, y=missing.index, color=sns.color_palette("crest")[3], ax=ax)
ax.set_xlabel("Fraction missing")
ax.set_title("Missingness across curated features")
fig.tight_layout()
fig.savefig(figure_path("missingness.png"), dpi=140)
plt.show()

## Feature distributions stratified by mortality

In [ ]:
interesting = ["age", "apache_4a_hospital_death_prob", "gcs_motor_apache", "creatinine_apache"]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.ravel(), interesting):
    if col not in df.columns:
        continue
    for label, sub in df.groupby(TARGET):
        sns.kdeplot(sub[col].dropna(), label=f"death={label}", fill=True, alpha=0.4, ax=ax)
    ax.set_title(col)
    ax.legend()
fig.suptitle("Feature distributions by hospital_death", y=1.02)
fig.tight_layout()
fig.savefig(figure_path("feature_distributions.png"), dpi=140)
plt.show()

## Correlation heatmap (numeric features)

In [ ]:
num_cols = [c for c in NUMERIC_FEATURES if c in df.columns]
corr = df[num_cols + [TARGET]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, square=True, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Numeric-feature correlation")
fig.tight_layout()
fig.savefig(figure_path("correlation_heatmap.png"), dpi=140)
plt.show()

**Takeaways**

* The dataset is heavily imbalanced — we need balanced metrics.
* ``apache_4a_hospital_death_prob`` is the dominant signal (the APACHE-IVa
  predicted probability is itself a clinical model output).
* GCS components and creatinine carry strong stratification.
* These same features dominate the SelectKBest top-K used for the
  quantum encoding (see notebook 04).